# 图像语义分割实验 - DeepLabv3

本实验使用 PyTorch 深度学习框架构建 DeepLabv3 网络模型，在 PASCAL VOC2012 数据集上进行图像语义分割。

本版本额外加入了适合实验报告展示的可视化与结果缓存功能：
- 训练过程指标自动保存到 JSON
- 无需重新训练即可重绘训练曲线
- 保存验证集定量评估结果
- 增加混淆矩阵、类别 IoU 柱状图、预测叠加图等可视化

## 步骤1：构建数据读取相关函数

In [1]:
import os
import json
import random
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet101
from torch.nn import CrossEntropyLoss
from torch.optim.lr_scheduler import _LRScheduler
from tqdm import tqdm

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 全局配置
IGNORE_LABEL = 255
NUM_CLASSES = 21
IMAGE_MEAN = [103.53, 116.28, 123.675]
IMAGE_STD = [57.375, 57.120, 58.395]
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = r'.\data\VOCdevkit\VOC2012'
BATCH_SIZE = 4
CROP_SIZE = 513
TRAIN_EPOCHS = 30
BASE_LR = 0.007
WEIGHT_DECAY = 0.0001
MOMENTUM = 0.9
SAVE_DIR = './model'
OUTPUT_STRIDE = 8
ASPP_ATROUS_RATES = [1, 6, 12, 18]
SEED = 42

VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog',
    'horse', 'motorbike', 'person', 'pottedplant', 'sheep',
    'sofa', 'train', 'tvmonitor'
]

VOC_COLORMAP = [
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0],
    [0, 0, 128], [128, 0, 128], [0, 128, 128], [128, 128, 128],
    [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
    [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128],
    [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0], [0, 64, 128]
]

Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f'当前设备: {DEVICE}')


当前设备: cpu


In [ ]:
plt.style.use("math.yaml")

In [ ]:
class VOC2012SegDataset(Dataset):
    """VOC2012 语义分割数据集（训练：随机增强；验证：短边缩放+中心裁剪）"""
    def __init__(self, data_root, split='train', crop_size=513, scale_range=(0.5, 2.0), flip_prob=0.5):
        self.data_root = data_root
        self.split = split
        self.crop_size = crop_size
        self.scale_min, self.scale_max = scale_range
        self.flip_prob = flip_prob
        self.ignore_label = IGNORE_LABEL
        self.num_classes = NUM_CLASSES

        self.img_dir = os.path.join(data_root, 'JPEGImages')
        self.anno_color_dir = os.path.join(data_root, 'SegmentationClass')
        self.anno_gray_dir = os.path.join(data_root, 'SegmentationClassGray')

        if not os.path.exists(self.anno_gray_dir):
            os.makedirs(self.anno_gray_dir)
            self._color2gray()

        split_file = os.path.join(data_root, f'ImageSets/Segmentation/{split}.txt')
        with open(split_file, 'r', encoding='utf-8') as f:
            self.img_ids = [line.strip() for line in f.readlines()]

        self.normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGE_MEAN, std=IMAGE_STD)
        ])

    def _color2gray(self):
        color2id = {
            (0, 0, 0): 0, (128, 0, 0): 1, (0, 128, 0): 2, (128, 128, 0): 3,
            (0, 0, 128): 4, (128, 0, 128): 5, (0, 128, 128): 6, (128, 128, 128): 7,
            (64, 0, 0): 8, (192, 0, 0): 9, (64, 128, 0): 10, (192, 128, 0): 11,
            (64, 0, 128): 12, (192, 0, 128): 13, (64, 128, 128): 14, (192, 128, 128): 15,
            (0, 64, 0): 16, (128, 64, 0): 17, (0, 192, 0): 18, (128, 192, 0): 19,
            (0, 64, 128): 20
        }
        print('开始将彩色标签图转为灰度标签图...')
        for img_name in os.listdir(self.anno_color_dir):
            color_img = Image.open(os.path.join(self.anno_color_dir, img_name)).convert('RGB')
            color_np = np.array(color_img)
            gray_np = np.ones_like(color_np[:, :, 0], dtype=np.uint8) * self.ignore_label
            for color, idx in color2id.items():
                mask = (color_np == color).all(axis=2)
                gray_np[mask] = idx
            Image.fromarray(gray_np).save(os.path.join(self.anno_gray_dir, img_name))
        print('彩色标签图转灰度完成！')

    def _random_aug(self, img, label):
        h, w = img.shape[:2]
        scale = np.random.uniform(self.scale_min, self.scale_max)
        new_h, new_w = int(h * scale), int(w * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
        label = cv2.resize(label, (new_w, new_h), interpolation=cv2.INTER_NEAREST)

        pad_h = max(self.crop_size - new_h, 0)
        pad_w = max(self.crop_size - new_w, 0)
        if pad_h > 0 or pad_w > 0:
            img = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            label = cv2.copyMakeBorder(label, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=self.ignore_label)

        crop_h = np.random.randint(0, img.shape[0] - self.crop_size + 1)
        crop_w = np.random.randint(0, img.shape[1] - self.crop_size + 1)
        img = img[crop_h:crop_h+self.crop_size, crop_w:crop_w+self.crop_size, :]
        label = label[crop_h:crop_h+self.crop_size, crop_w:crop_w+self.crop_size]

        if np.random.uniform(0, 1) < self.flip_prob:
            img = cv2.flip(img, 1)
            label = cv2.flip(label, 1)
        return img, label

    def _val_resize_crop(self, img, label):
        """验证模式：保持宽高比缩放使短边=crop_size，再中心裁剪。"""
        h, w = img.shape[:2]
        if h < w:
            new_h = self.crop_size
            new_w = int(round(w * self.crop_size / h))
        else:
            new_w = self.crop_size
            new_h = int(round(h * self.crop_size / w))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
        label = cv2.resize(label, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
        top = (new_h - self.crop_size) // 2
        left = (new_w - self.crop_size) // 2
        img = img[top:top+self.crop_size, left:left+self.crop_size, :]
        label = label[top:top+self.crop_size, left:left+self.crop_size]
        return img, label

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_path = os.path.join(self.img_dir, f'{img_id}.jpg')
        label_path = os.path.join(self.anno_gray_dir, f'{img_id}.png')

        img_pil = Image.open(img_path).convert('RGB')
        img = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        label = np.array(Image.open(label_path).convert('L'), dtype=np.uint8)

        if 'train' in self.split:
            img, label = self._random_aug(img, label)
        else:
            img, label = self._val_resize_crop(img, label)

        img = self.normalize(img.astype(np.float32))
        label = torch.tensor(label.astype(np.int64))
        return img, label

    def __len__(self):
        return len(self.img_ids)

def get_dataloader(data_root, split='train', batch_size=16, crop_size=CROP_SIZE, num_workers=0):
    dataset = VOC2012SegDataset(data_root, split, crop_size)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(split == 'train'),
        num_workers=num_workers,
        pin_memory=True,
        drop_last=(split == 'train')
    )
    return dataloader


In [ ]:
import torch.multiprocessing
torch.multiprocessing.set_sharing_strategy('file_system')

train_loader = get_dataloader(DATA_ROOT, 'train', batch_size=2, num_workers=0)
img, label = next(iter(train_loader))
print(f'图像张量形状：{img.shape}，标签张量形状：{label.shape}')
print(f'图像数据类型：{img.dtype}，标签数据类型：{label.dtype}')
print('✅ 数据集加载成功！')

## 步骤2：构建 DeepLabv3 网络

In [ ]:
class ASPPConv(nn.Module):
    def __init__(self, in_channels, out_channels, atrous_rate):
        super().__init__()
        if atrous_rate == 1:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=atrous_rate, dilation=atrous_rate, bias=False)
        self.conv = nn.Sequential(conv, nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.conv(x)

class ASPPPooling(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[2:]
        x = self.pool(x)
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)

class ASPP(nn.Module):
    def __init__(self, in_channels=2048, out_channels=256):
        super().__init__()
        self.conv1 = ASPPConv(in_channels, out_channels, ASPP_ATROUS_RATES[0])
        self.conv2 = ASPPConv(in_channels, out_channels, ASPP_ATROUS_RATES[1])
        self.conv3 = ASPPConv(in_channels, out_channels, ASPP_ATROUS_RATES[2])
        self.conv4 = ASPPConv(in_channels, out_channels, ASPP_ATROUS_RATES[3])
        self.pool = ASPPPooling(in_channels, out_channels)
        self.fuse = nn.Sequential(
            nn.Conv2d(out_channels * 5, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        x = torch.cat([self.conv1(x), self.conv2(x), self.conv3(x), self.conv4(x), self.pool(x)], dim=1)
        return self.fuse(x)

class DeepLabV3(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, output_stride=OUTPUT_STRIDE, pretrained=True):
        super().__init__()
        self.num_classes = num_classes
        self.output_stride = output_stride

        resnet = resnet101(pretrained=pretrained)
        self.backbone_layer1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.backbone_layer2 = resnet.layer1
        self.backbone_layer3 = resnet.layer2
        self.backbone_layer4 = resnet.layer3
        self.backbone_layer5 = resnet.layer4

        if self.output_stride == 8:
            self._modify_backbone_stride_dilation(self.backbone_layer4, stride=1, dilation=2)
            self._modify_backbone_stride_dilation(self.backbone_layer5, stride=1, dilation=4)
        elif self.output_stride == 16:
            self._modify_backbone_stride_dilation(self.backbone_layer5, stride=1, dilation=2)

        self.aspp = ASPP(in_channels=2048, out_channels=256)
        self.classifier = nn.Conv2d(256, num_classes, kernel_size=1)

    def _modify_backbone_stride_dilation(self, layer, stride, dilation):
        for m in layer.modules():
            if isinstance(m, nn.Conv2d):
                if m.stride == (2, 2):
                    m.stride = (stride, stride)
                if m.dilation == (1, 1) and m.kernel_size == (3, 3):
                    m.dilation = (dilation, dilation)
                    m.padding = (dilation, dilation)

    def forward(self, x):
        size = x.shape[2:]
        x = self.backbone_layer1(x)
        x = self.backbone_layer2(x)
        x = self.backbone_layer3(x)
        x = self.backbone_layer4(x)
        x = self.backbone_layer5(x)
        x = self.aspp(x)
        x = self.classifier(x)
        return F.interpolate(x, size=size, mode='bilinear', align_corners=False)

## 步骤3：定义学习率调度器

In [ ]:
class PolyLR(_LRScheduler):
    def __init__(self, optimizer, total_steps, end_lr=0.0, power=0.9, last_epoch=-1):
        self.total_steps = total_steps
        self.end_lr = end_lr
        self.power = power
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = min(self.last_epoch, self.total_steps)
        lr = (self.base_lrs[0] - self.end_lr) * ((1 - step / self.total_steps) ** self.power) + self.end_lr
        return [lr for _ in self.base_lrs]

## 步骤4：定义损失函数

In [ ]:
class SegCrossEntropyLoss(nn.Module):
    def __init__(self, ignore_label=IGNORE_LABEL):
        super().__init__()
        self.ignore_label = ignore_label

    def forward(self, logits, labels):
        return F.cross_entropy(logits, labels, ignore_index=self.ignore_label, reduction='mean')

## 步骤5：构建训练网络并执行训练

In [ ]:
def freeze_bn(model):
    for m in model.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            m.weight.requires_grad = False
            m.bias.requires_grad = False
            m.track_running_stats = True

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def build_model(num_classes=NUM_CLASSES):
    """使用本文件 Cell 7 中自定义的 DeepLabV3（ResNet101 + ASPP）。"""
    return DeepLabV3(num_classes=num_classes, output_stride=OUTPUT_STRIDE, pretrained=True)

def train():
    torch.cuda.empty_cache()
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

    print('加载 VOC2012 数据集...')
    train_loader = get_dataloader(DATA_ROOT, 'train', BATCH_SIZE, CROP_SIZE, num_workers=0)
    total_steps = len(train_loader) * TRAIN_EPOCHS
    print(f'训练总步数：{total_steps}，每轮步数：{len(train_loader)}')

    print('初始化 DeepLabv3 模型...')
    model = build_model().to(DEVICE)
    # 小 batch 下固定 BN，避免统计量抖动导致训练崩溃
    freeze_bn(model)

    criterion = SegCrossEntropyLoss().to(DEVICE)
    optimizer = optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY
    )
    scheduler = PolyLR(optimizer, total_steps=total_steps, power=0.9)

    history = {
        'config': {
            'data_root': DATA_ROOT,
            'batch_size': BATCH_SIZE,
            'crop_size': CROP_SIZE,
            'train_epochs': TRAIN_EPOCHS,
            'base_lr': BASE_LR,
            'weight_decay': WEIGHT_DECAY,
            'momentum': MOMENTUM,
            'output_stride': OUTPUT_STRIDE,
            'device': str(DEVICE)
        },
        'epoch_loss': [],
        'epoch_lr': [],
        'step_loss': [],
        'step_lr': []
    }

    model.train()
    print(f'开始训练，共 {TRAIN_EPOCHS} 轮，设备：{DEVICE}')
    for epoch in range(TRAIN_EPOCHS):
        epoch_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Epoch {epoch+1}/{TRAIN_EPOCHS}')
        for step, (img, label) in pbar:
            img = img.to(DEVICE, non_blocking=True)
            label = label.to(DEVICE, non_blocking=True).long()

            logits = model(img)
            loss = criterion(logits, label)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            current_loss = float(loss.item())
            current_lr = float(scheduler.get_last_lr()[0])
            epoch_loss += current_loss
            history['step_loss'].append(current_loss)
            history['step_lr'].append(current_lr)

            pbar.set_postfix({'Step Loss': f'{current_loss:.4f}', 'LR': f'{current_lr:.6f}'})

        avg_loss = epoch_loss / len(train_loader)
        history['epoch_loss'].append(float(avg_loss))
        history['epoch_lr'].append(float(scheduler.get_last_lr()[0]))
        print(f'Epoch {epoch+1} 平均损失：{avg_loss:.4f}')

    model_name = f'deeplabv3_custom_os{OUTPUT_STRIDE}_e{TRAIN_EPOCHS}.pth'
    model_path = os.path.join(SAVE_DIR, model_name)
    history_path = os.path.join(SAVE_DIR, 'training_history.json')

    torch.save(model.state_dict(), model_path)
    history['model_path'] = model_path
    save_json(history, history_path)

    print(f'模型训练完成，已保存至：{model_path}')
    print(f'训练过程数据已保存至：{history_path}')
    return model_path, history_path


In [ ]:
# 首次运行时执行训练；后续如果已有模型和 JSON，可跳过本单元
torch.backends.cudnn.benchmark = False
model_path, history_path = train()

## 训练过程可视化（从 JSON 读取，无需重新训练）

In [ ]:
history_path = os.path.join(SAVE_DIR, 'training_history.json')
history = load_json(history_path)
print('已读取训练历史文件：', history_path)
print('可用字段：', list(history.keys()))

In [ ]:
def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

    axes[0].plot(range(1, len(history['epoch_loss']) + 1), history['epoch_loss'], marker='o', linewidth=2, color='#1f77b4')
    axes[0].set_title('训练损失 - Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')

    axes[1].plot(range(1, len(history['step_loss']) + 1), history['step_loss'], linewidth=1.3, color='#d62728', alpha=0.9, label='Step Loss')
    if len(history['step_loss']) >= 5:
        kernel = min(15, len(history['step_loss']))
        smooth = np.convolve(history['step_loss'], np.ones(kernel) / kernel, mode='valid')
        axes[1].plot(range(kernel, kernel + len(smooth)), smooth, linewidth=2.0, color='#2ca02c', label='平滑损失')
    axes[1].set_title('训练损失 - Step')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'training_loss_curves.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('训练损失曲线已保存至：', save_path)

def plot_lr_curve(history):
    plt.figure(figsize=(8, 5), dpi=150)
    plt.plot(range(1, len(history['step_lr']) + 1), history['step_lr'], color='#9467bd', linewidth=2)
    plt.title('学习率变化曲线')
    plt.xlabel('Step')
    plt.ylabel('Learning Rate')
    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'learning_rate_curve.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('学习率曲线已保存至：', save_path)

plot_training_curves(history)
plot_lr_curve(history)

## 步骤6：验证网络（学生自主完成部分）

In [ ]:
class SegmentationMetrics:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.confusion_matrix = np.zeros((num_classes, num_classes), dtype=np.float64)

    def update(self, pred, label):
        mask = (label >= 0) & (label < self.num_classes)
        label = label[mask].astype(np.int64)
        pred = pred[mask].astype(np.int64)
        hist = np.bincount(self.num_classes * label + pred, minlength=self.num_classes ** 2)
        self.confusion_matrix += hist.reshape(self.num_classes, self.num_classes)

    def get_metrics(self):
        hist = self.confusion_matrix
        pa = np.diag(hist).sum() / np.maximum(hist.sum(), 1)
        class_acc = np.diag(hist) / np.maximum(hist.sum(axis=1), 1)
        iou = np.diag(hist) / np.maximum(hist.sum(axis=1) + hist.sum(axis=0) - np.diag(hist), 1)
        freq = hist.sum(axis=1) / np.maximum(hist.sum(), 1)
        fw_iou = (freq[freq > 0] * iou[freq > 0]).sum()
        return {
            'PA': float(pa),
            'MPA': float(np.nanmean(class_acc)),
            'mIoU': float(np.nanmean(iou)),
            'FWIoU': float(fw_iou),
            'class_acc': class_acc.tolist(),
            'IoU': iou.tolist(),
            'confusion_matrix': hist.tolist()
        }

def denormalize_image(img_tensor):
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    img = img * np.array(IMAGE_STD) + np.array(IMAGE_MEAN)
    img = np.clip(img, 0, 255).astype(np.uint8)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def label_to_color(label):
    h, w = label.shape
    color_img = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        color_img[label == i] = VOC_COLORMAP[i]
    return color_img

def overlay_mask(image, color_mask, alpha=0.45):
    return np.clip((1 - alpha) * image + alpha * color_mask, 0, 255).astype(np.uint8)

In [ ]:
def evaluate(model, val_loader, device, save_json_path=None):
    model.eval()
    metrics = SegmentationMetrics(NUM_CLASSES)

    with torch.no_grad():
        for img, label in tqdm(val_loader, desc='验证中'):
            img = img.to(device)
            label_np = label.numpy()
            outputs = model(img)
            if isinstance(outputs, dict):
                outputs = outputs['out']
            pred = torch.argmax(outputs, dim=1).cpu().numpy()
            for p, l in zip(pred, label_np):
                metrics.update(p.flatten(), l.flatten())

    results = metrics.get_metrics()
    print('\n评估结果：')
    print(f"像素准确率 (PA): {results['PA']:.4f}")
    print(f"平均像素准确率 (MPA): {results['MPA']:.4f}")
    print(f"平均交并比 (mIoU): {results['mIoU']:.4f}")
    print(f"频权交并比 (FWIoU): {results['FWIoU']:.4f}")

    if save_json_path is not None:
        save_json(results, save_json_path)
        print('评估结果已保存至：', save_json_path)
    return results

In [ ]:
# 加载训练好的模型并评估
history = load_json(os.path.join(SAVE_DIR, 'training_history.json'))
model_path = history.get('model_path')

model = build_model()
state_dict = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)

val_loader = get_dataloader(DATA_ROOT, 'val', batch_size=1, crop_size=CROP_SIZE, num_workers=0)
eval_json_path = os.path.join(SAVE_DIR, 'eval_results.json')
results = evaluate(model, val_loader, DEVICE, save_json_path=eval_json_path)


## 定量结果可视化

In [ ]:
results = load_json(os.path.join(SAVE_DIR, 'eval_results.json'))
print('已读取评估结果文件。')

In [ ]:
def plot_metric_summary(results):
    metric_names = ['PA', 'MPA', 'mIoU', 'FWIoU']
    metric_values = [results[k] for k in metric_names]

    plt.figure(figsize=(8, 5), dpi=150)
    bars = plt.bar(metric_names, metric_values, color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'])
    plt.ylim(0, 1)
    plt.ylabel('数值')
    plt.title('分割评估核心指标')
    for bar, value in zip(bars, metric_values):
        plt.text(bar.get_x() + bar.get_width()/2, value + 0.02, f'{value:.4f}', ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'metric_summary.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('指标汇总图已保存至：', save_path)

def plot_per_class_iou(results):
    iou = np.array(results['IoU'])
    order = np.argsort(iou)[::-1]

    plt.figure(figsize=(12, 6), dpi=150)
    plt.bar(np.array(VOC_CLASSES)[order], iou[order], color='#4C9F70')
    plt.xticks(rotation=60, ha='right')
    plt.ylim(0, 1)
    plt.ylabel('IoU')
    plt.title('各类别 IoU 柱状图')
    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'per_class_iou.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('类别 IoU 图已保存至：', save_path)

plot_metric_summary(results)
plot_per_class_iou(results)

In [ ]:
def plot_confusion_matrix_heatmap(results):
    cm = np.array(results['confusion_matrix'], dtype=np.float64)
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    plt.figure(figsize=(12, 10), dpi=150)
    sns.heatmap(cm_norm, cmap='YlGnBu', square=False, cbar=True, xticklabels=VOC_CLASSES, yticklabels=VOC_CLASSES)
    plt.title('归一化混淆矩阵')
    plt.xlabel('预测类别')
    plt.ylabel('真实类别')
    plt.xticks(rotation=60, ha='right', fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'confusion_matrix_heatmap.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('混淆矩阵热力图已保存至：', save_path)

plot_confusion_matrix_heatmap(results)

## 定性可视化：分割结果展示

In [ ]:
def visualize_results(model, val_loader, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples), dpi=150)
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    with torch.no_grad():
        for idx, (img, label) in enumerate(val_loader):
            if idx >= num_samples:
                break

            outputs = model(img.to(device))
            if isinstance(outputs, dict):
                outputs = outputs['out']
            pred = torch.argmax(outputs, dim=1).cpu().numpy()[0]

            image = denormalize_image(img[0])
            label_color = label_to_color(label[0].numpy())
            pred_color = label_to_color(pred)

            axes[idx, 0].imshow(image)
            axes[idx, 0].set_title('原图')
            axes[idx, 0].axis('off')

            axes[idx, 1].imshow(label_color)
            axes[idx, 1].set_title('真实标签')
            axes[idx, 1].axis('off')

            axes[idx, 2].imshow(pred_color)
            axes[idx, 2].set_title('预测结果')
            axes[idx, 2].axis('off')

    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'segmentation_triplets.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('三联图已保存至：', save_path)

visualize_results(model, val_loader, DEVICE, num_samples=3)

In [ ]:
def visualize_overlay_results(model, val_loader, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples), dpi=150)
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    with torch.no_grad():
        for idx, (img, label) in enumerate(val_loader):
            if idx >= num_samples:
                break

            outputs = model(img.to(device))
            if isinstance(outputs, dict):
                outputs = outputs['out']
            pred = torch.argmax(outputs, dim=1).cpu().numpy()[0]

            image = denormalize_image(img[0])
            label_color = label_to_color(label[0].numpy())
            pred_color = label_to_color(pred)

            axes[idx, 0].imshow(image)
            axes[idx, 0].set_title('原图')
            axes[idx, 0].axis('off')

            axes[idx, 1].imshow(overlay_mask(image, label_color, alpha=0.45))
            axes[idx, 1].set_title('真实标签叠加图')
            axes[idx, 1].axis('off')

            axes[idx, 2].imshow(overlay_mask(image, pred_color, alpha=0.45))
            axes[idx, 2].set_title('预测结果叠加图')
            axes[idx, 2].axis('off')

    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'segmentation_overlay.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('叠加图已保存至：', save_path)

visualize_overlay_results(model, val_loader, DEVICE, num_samples=3)

In [ ]:
def visualize_error_map(model, val_loader, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(18, 5 * num_samples), dpi=150)
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    with torch.no_grad():
        for idx, (img, label) in enumerate(val_loader):
            if idx >= num_samples:
                break

            outputs = model(img.to(device))
            if isinstance(outputs, dict):
                outputs = outputs['out']
            pred = torch.argmax(outputs, dim=1).cpu().numpy()[0]
            gt = label[0].numpy()
            err = (pred != gt).astype(np.uint8)

            image = denormalize_image(img[0])
            axes[idx, 0].imshow(image)
            axes[idx, 0].set_title('原图')
            axes[idx, 0].axis('off')

            axes[idx, 1].imshow(label_to_color(gt))
            axes[idx, 1].set_title('真实标签')
            axes[idx, 1].axis('off')

            axes[idx, 2].imshow(label_to_color(pred))
            axes[idx, 2].set_title('预测结果')
            axes[idx, 2].axis('off')

            axes[idx, 3].imshow(err, cmap='Reds')
            axes[idx, 3].set_title(f'误差图，错误像素占比：{err.mean():.2%}')
            axes[idx, 3].axis('off')

    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'segmentation_error_map.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('误差图已保存至：', save_path)

visualize_error_map(model, val_loader, DEVICE, num_samples=3)

## 可选：展示数据增强前后的训练样本

In [ ]:
def show_dataset_samples(data_root, sample_index=0):
    raw_dataset = VOC2012SegDataset(data_root, split='train', crop_size=CROP_SIZE)
    img_tensor, label = raw_dataset[sample_index]
    image = denormalize_image(img_tensor)
    label_color = label_to_color(label.numpy())

    plt.figure(figsize=(10, 4), dpi=150)
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title('训练样本图像')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(label_color)
    plt.title('对应标签')
    plt.axis('off')

    plt.tight_layout()
    save_path = os.path.join(SAVE_DIR, 'dataset_sample.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print('样本展示图已保存至：', save_path)

show_dataset_samples(DATA_ROOT, sample_index=0)

## 实验总结

本实验主要介绍如何使用 PyTorch 框架在 VOC2012 数据集上训练和推理 DeepLabv3 网络模型，从而实现图像语义分割任务。

通过本实验学员将了解：
- 如何处理图像分割数据标签
- 如何定义和训练卷积神经网络
- 如何评估语义分割模型性能
- 如何利用 JSON 持久化训练过程数据
- 如何绘制训练曲线、混淆矩阵、类别 IoU 图与分割结果可视化